# Setup

In [1]:
# Basics
import requests 
import os
import io
import docx
import sys
import json
import warnings
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import time
import psycopg
from concurrent.futures import ThreadPoolExecutor
# git source libraries
import gitsource
from gitsource import GithubRepositoryDataReader
from gitsource import chunk_documents
from dotenv import load_dotenv

# Neural Network libraries
import torch

# LLM
import openai
from openai import OpenAI

# Search libraries
import minsearch
from minsearch import VectorSearch
from minsearch import Index
from sentence_transformers import SentenceTransformer
from sqlitesearch import VectorSearchIndex
from elasticsearch import Elasticsearch

# sklearn libraries
import sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
from sklearn.decomposition import NMF

# toyaikit
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat.runners import OpenAIResponsesRunner


In [2]:
load_dotenv("/workspace/.env")
openai_client = OpenAI()

# Generating Ground Truth Data

To evaluate whether retrieval actually recovers the correct documents, we need a set of (question, expected document) pairs with a known ground truth.

We cannot use the original `question` field from the FAQ as our test set. Those questions live literally inside the documents, so any retrieval hits them trivially. It would be like grading a test with the answer key open.

We use the LLM in reverse: instead of answering questions, it pretends to be a student and generates new questions that could lead to each document. Each original document produces 5 paraphrases, questions that should retrieve that document but written with words different from the answer.

Each document is a dictionary with keys `course`, `question`, and `text`. The generation function receives the list of documents and the OpenAI client, and returns a dictionary where each key is the doc ID and the value is the list of generated questions for that document.

In [3]:
# From Module 1


def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents


def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index

## Loading the documents

In [4]:
def load_faq_data():
    docs_url = 'https://datatalks.club/faq/json/courses.json'
    response = requests.get(docs_url)
    courses_raw = response.json()

    documents = []
    url_prefix = 'https://datatalks.club/faq'

    for course in courses_raw:
        course_url = f'{url_prefix}{course["path"]}'
        course_response = requests.get(course_url)
        course_response.raise_for_status()
        course_data = course_response.json()

        documents.extend(course_data)

    return documents


In [5]:
documents = load_faq_data()

In [6]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

113

In [7]:
documents = documents_llm

In [8]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


The ID becomes the label in our ground truth dataset. We generate questions from a document, so we know that this document holds the answer. Later, search evaluation checks whether search brings back the document with this ID.

This is why every record needs a stable ID.

## Generating Questions with structured output

Instead of getting a paragraph that contains questions, we can ask for a Python object with a questions field.

This is useful when code will process the output. The model returns the same structure every time. We can access the generated questions directly instead of parsing text manually.

We want the output as a list of strings, so we define that structure with a Pydantic model:

In [9]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [10]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long and not use —.
""".strip()

In [11]:
user_prompt = json.dumps(doc)

In [12]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [13]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [14]:
result = response.output_parsed

print(result)

questions=['I just found this course, can I still join it and take part now?', 'Is it okay to start the course late if I just discovered it?', 'Can I still participate in the course even if I missed the beginning?', 'If I join the course now, do I still have a chance to get a certificate?', "What do I need to do to qualify for the certificate if I'm joining late?"]


## Utility functions

In [15]:
def llm_structured(client, instructions, user_prompt, output_type, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage

result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['Can I still join the course if I just found it now?', 'Is it too late to start llm-zoomcamp after the course has already begun?', 'If I join late, can I still get a certificate?', 'What do I need to do to earn the certificate if I start after the course starts?', 'Are project submissions still open for people who join the course now?']


In [16]:
usage.input_tokens, usage.output_tokens

(211, 90)

In [17]:
def calc_price(usage):
    input_price_per_million = 0.75
    output_price_per_million = 4.50

    input_cost = (usage.input_tokens / 1_000_000) * input_price_per_million
    output_cost = (usage.output_tokens / 1_000_000) * output_price_per_million
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }


cost = calc_price(usage)

cost

{'input_cost': 0.00015825,
 'output_cost': 0.00040500000000000003,
 'total_cost': 0.0005632500000000001}

In [18]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'Can I still join the course if I just found it now?',
  'document': '74eb249bbf'},
 {'question': 'Is it too late to start llm-zoomcamp after the course has already begun?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'What do I need to do to earn the certificate if I start after the course starts?',
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for people who join the course now?',
  'document': '74eb249bbf'}]

Each record has two fields:

- question: the question generated by the LLM
- document: the ID of the FAQ document that should answer the question

The document field connects the generated question to the document that contains the answer. Later, when we evaluate search, we'll ask the search engine the generated question. Then we'll check if it retrieves the document with this ID.

## Ground Truth Batch

The processing function takes one document and turns it into ground truth records.

For each document, we:

convert the document to JSON so we can send it to the LLM
ask the LLM to return a Questions object
create one ground truth record for each generated question
Each record contains the generated question and the ID of the document that should answer the question.

When we send many requests, one of them might fail. We don't want the entire batch to fail because of one temporary error.

In [19]:
def llm_structured_retry(
    client,
    instructions,
    user_prompt,
    output_type,
    model="gpt-5.4-mini",
    max_retries=3,
):
    for attempt in range(max_retries):
        try:
            return llm_structured(
                client,
                instructions,
                user_prompt,
                output_type,
                model=model,
            )
        except Exception:
            if attempt == max_retries - 1:
                raise
            time.sleep(2 ** attempt)


llm_structured makes one structured-output call. llm_structured_retry wraps the same call in a retry loop. If one request fails because of a temporary API or network issue, it waits briefly and tries again.

In [20]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [21]:
# from tqdm.auto import tqdm

# ground_truth = []
# usages = []

# for doc in tqdm(documents[:5]):
#     records, usage = generate_ground_truth(doc)
#     ground_truth.extend(records)
#     usages.append(usage)

This works, but it runs one LLM call after another. Running it for all documents this way would take too long.

### Parallel processing

In [22]:

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results

In [23]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, documents, generate_ground_truth)

With 5 questions per document, you should get roughly 5x the number of documents.

Calculate the total cost:

In [24]:
# ground_truth = []
# usages = []

# for records, usage in results:
#     ground_truth.extend(records)
#     usages.append(usage)

# len(ground_truth)

In [25]:
# total_cost = 0.0

# for usage in usages:
#     cost = calc_price(usage)
#     total_cost = total_cost + cost["total_cost"]

# total_cost

In [26]:
# calc_total_price(usages)

Create a dataframe so we can look at the records as a table and save them as a CSV file.

In [27]:
# df_ground_truth = pd.DataFrame(ground_truth)

In [28]:
# df_ground_truth.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/ground_truth-new.csv", index=False)

# Search Evaluation

For search evaluation, we need a dataset of questions where we know which document is the correct answer. We'll use an LLM to generate these questions from our FAQ data.

The approach works like this:

- A = the original answer in the FAQ
- Q* = a question generated from that answer by an LLM
- We send Q* through our search and check if the original document appears in the results

For RAG evaluation, we go one step further:

- A = the original answer in the FAQ
- Q* = a question generated from that answer by an LLM
- A' = the answer produced by our RAG system when given Q*
- We compare A' with A to see if the system produced the right answer

This is the A → Q* → A' pattern. We know the answer for each generated question because we created the question from that answer.

In [29]:
# pair list of questions with their corresponding document IDs
df_ground_truth = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [30]:
ground_truth[0]

{'question': 'Can I still join the course if I just found out about it?',
 'document': '74eb249bbf'}

Load the documents and build a minsearch index:

In [31]:
def build_index(documents):
    index = Index(
        text_fields=['question', 'section', 'answer'],
        keyword_fields=['course']
    )
    index.fit(documents)
    return index


documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [32]:
documents[0]

{'id': '74eb249bbf',
 'course': 'llm-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'I just discovered the course. Can I still join?',
 'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'}

In [33]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

using the first gorund truth record to test the search function, we can check if the document ID returned by the search matches the expected document ID. 

In [34]:
q = ground_truth[0]
q

{'question': 'Can I still join the course if I just found out about it?',
 'document': '74eb249bbf'}

In [35]:
doc_id = q["document"]
results = text_search(query=q["question"])
results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': 'a9353fadfe',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'The homework submission form is still open even though the deadline has passed — can I still submit?',
  'answer': "Yes. As long as the submission form is still open, you can submit your answers, even if the listed deadline has already passed. You can no longer submit only after the form has been closed — so while it's still open, go ahead and submit."},
 {'id': '9f689c185f',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I missed the first homework - can I still get a certificate?',
  'answer': 'Yes, you need to pass the Capstone proj

In [36]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

74eb249bbf == 74eb249bbf: True
a9353fadfe == 74eb249bbf: False
9f689c185f == 74eb249bbf: False
977bf7786c == 74eb249bbf: False
69d122f12e == 74eb249bbf: False


From this we can construct a vector of 0s and 1s, where 1 means the search returned the correct document and 0 means it did not. The mean of this vector is the accuracy of the search on this dataset. All thoses vector is then stored in a matrix, where each row is a question and each column is a search. The mean of each column is the accuracy of that search engine on this dataset.

In [37]:
# Wraping the above code in a function
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [38]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total(ground_truth_sample, text_search)

relevance_total_text

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 1, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

**Hit Rate (HR) or Recall at k**

Measures the proportion of queries for which at least one relevant document is retrieved in the top k results. Hit Rate tells us if we found the right document, but not where it was.

$$
\mathrm{HR@k} = \frac{\text{Number of queries with at least one relevant document in top } k}{|\mathcal{Q}|}
$$

In [39]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

hit_rate(relevance_total_text)

0.8666666666666667

**Mean Reciprocal Rank (MRR)**

Evaluates the rank position of the first relevant document. It also consider positions as in the Hit Rate, but it gives more weight to documents that are ranked higher. The reciprocal rank is the multiplicative inverse of the rank of the first relevant document. The mean reciprocal rank is the average of the reciprocal ranks for a set of queries.

$$
\mathrm{MRR} = \frac{1}{|\mathcal{Q}|} \sum_{q \in \mathcal{Q}} \frac{1}{\mathrm{rank}_q}
$$


For each query, the score is based on the rank of the first correct document, for example:

- position 1: score is 1.0
- position 2: score is 0.5
- position 3: score is 0.333
- not found: score is 0

In [40]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

mrr(relevance_total_text)

0.6577777777777778

Putting the metrics in a single function, we can evaluate the search results for a set of queries and their expected relevant documents. 

In [41]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

# Now evaluating for all ground truth
evaluate(ground_truth, text_search)

  0%|          | 0/565 [00:00<?, ?it/s]

{'hit_rate': 0.8530973451327434, 'mrr': 0.7227433628318578}

Our ground truth assumes only one relevant document per query. In practice, other retrieved documents might also be relevant. A 50% hit rate does not mean that half the results are useless. It means the document we generated the question from did not appear in the top results for half the queries. Other relevant documents may still be there.

With synthetic data, the generated questions can be too close to the original FAQ text. This inflates hit rate and MRR. If you see numbers above 95%, treat them with caution and check whether the questions are realistic enough.

Good thresholds depend on your use case. A 50% hit rate is acceptable for some applications, while others need 90% or higher. The right number depends on how much the downstream LLM can compensate for imperfect retrieval. It also depends on user tolerance for wrong answers.


In [42]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [43]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/565 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/565 [00:00<?, ?it/s]

Search parameters can look arbitrary. This includes field boosts, number of results, filters, and other settings. Evaluation gives us a way to compare settings with evidence.

Grid search is fine when there are only a few settings. For a larger parameter space, use a smarter search strategy. You can sample random combinations, use Bayesian optimization, or keep a validation split so you don't overfit the evaluation set.

For text search on our dataset, grid search takes about one second per combination. That makes it practical to try many options. When each evaluation takes minutes instead of seconds, grid search becomes too expensive. In those cases, use Bayesian optimization with a library like hyperopt.

In [44]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.980531,0.878732
19,2.0,4.0,0.2,0.980531,0.878732
35,5.0,10.0,0.5,0.980531,0.878732
6,1.0,4.0,0.1,0.973451,0.878024
7,1.0,4.0,0.2,0.975221,0.877611
4,1.0,2.0,0.2,0.980531,0.876224
18,2.0,4.0,0.1,0.978761,0.875929
33,5.0,10.0,0.1,0.978761,0.875782
34,5.0,10.0,0.2,0.978761,0.875782
8,1.0,4.0,0.5,0.964602,0.874041


In [45]:
# Define the search function with these boosts params
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

Hit Rate tells us whether the correct document appears at all. MRR tells us whether it appears near the top. A document near the top is more likely to be used by the RAG prompt.

We return 5 results from search. Increasing top-K to 10 would improve hit rate because there are more chances to find the correct document. But more results means more context sent to the LLM. That costs more and makes it harder for the model to identify what is relevant. Five results is a reasonable default for short FAQ-style documents.

# Generating RAG Answers

Until now we've evaluated search in isolation. We generated questions from the FAQ and checked whether search could find the original document.

- Query -> [Retrieval] -> Top-5 docs -> Hit or Miss?

This is the RAG pipeline at runtime: the user asks a question, the search engine retrieves documents, and the LLM uses those documents to answer. The evaluation above only checks the retrieval step. The next step is to check the whole pipeline, because good retrieval does not guarantee a good final answer.

- Query -> [Retrieval] -> Top-5 docs -> [Prompt] -> [LLM] -> Answer


In [46]:
df_ground_truth = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [47]:
documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [48]:
# Create a lookup table for the original FAQ documents:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

We did not need this during search evaluation because there we compared id with id (the `d["id"] == doc_id` check). Only the id mattered, not the full document.

In RAG evaluation we need the text of the original answer to send to the LLM judge alongside the generated answer. Since the text lives inside the full document, we access it by id through this lookup.


In [49]:

def calc_total_price(usages):
    total_cost = 0.0

    for usage in usages:
        cost = calc_price(usage)
        total_cost = total_cost + cost["total_cost"]

    return total_cost

# From Module 1 ------------------------------------------------
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

PROMPT_TEMPLATE = '''
QUESTION: {question}

CONTEXT:
{context}
'''.strip()


class RAGBase:

    def __init__(
        self,
        index,
        llm_client,
        instructions=INSTRUCTIONS,
        prompt_template=PROMPT_TEMPLATE,
        course='llm-zoomcamp',
        model='gpt-5.4-mini'
    ):
        self.index = index
        self.llm_client = llm_client
        self.instructions = instructions
        self.course = course
        self.prompt_template = prompt_template
        self.model = model

    def search(self, query, num_results=5):
        boost_dict = {'question': 3.0, 'section': 0.5}
        filter_dict = {'course': self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def build_context(self, search_results):
        lines = []

        for doc in search_results:
            lines.append(doc['section'])
            lines.append('Q: ' + doc['question'])
            lines.append('A: ' + doc['answer'])
            lines.append('')

        return '\n'.join(lines).strip()

    def build_prompt(self, query, search_results):
        context = self.build_context(search_results)
        return self.prompt_template.format(
            question=query, context=context
        )

    def llm(self, prompt):
        input_messages = [
            {'role': 'developer', 'content': self.instructions},
            {'role': 'user', 'content': prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        return response.output_text

    def rag(self, query):
        search_results = self.search(query)
        prompt = self.build_prompt(query, search_results)
        answer = self.llm(prompt)
        return answer
# From Module 1 ------------------------------------------------



class RAGWithUsage(RAGBase):

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.usages = []
        self.last_usage = None

    def reset_usage(self):
        self.usages = []
        self.last_usage = None

    def search(self, query, num_results=5):
        boost_dict = {"question": 1.0, "answer": 2.0, "section": 0.1}
        filter_dict = {"course": self.course}

        return self.index.search(
            query,
            num_results=num_results,
            boost_dict=boost_dict,
            filter_dict=filter_dict
        )

    def llm(self, prompt):
        input_messages = [
            {"role": "developer", "content": self.instructions},
            {"role": "user", "content": prompt}
        ]

        response = self.llm_client.responses.create(
            model=self.model,
            input=input_messages
        )

        self.last_usage = response.usage
        self.usages.append(response.usage)

        return response.output_text

    def total_cost(self):
        return calc_total_price(self.usages)

RAGWithUsage extends the RAGBase class from module 1 with two small changes so we can use it during evaluation.

First, it overrides the search method to use the tuned boosts we found during search tuning (`question=1.0`, `answer=2.0`, `section=0.1`). The original RAGBase used `question=3.0` and no boost on `answer`, which the grid search proved suboptimal for this dataset.

Second, it overrides the llm method to save the token usage of every call. Each response has a `usage` object with input, output, and total tokens. Storing them in a list gives us a full history that we can sum later.

Two helpers complete the picture. The `total_cost` method sums the whole usage history through `calc_total_price`. The `reset_usage` method clears the list, useful when we want to discard the tokens spent during sanity checks before running the full batch.

The pipeline itself (`rag = search -> build_prompt -> llm`) stays inherited from RAGBase. Python resolves `self.search` and `self.llm` on the subclass first, so the tuned search and instrumented llm plug in automatically. From the outside, `assistant.rag(query)` looks identical to the base class; the difference lives inside.


In [50]:
assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [51]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [52]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'Can I still join the course if I just found out about it?',
 'answer_llm': 'Yes, you can still join the course. If you want a certificate, make sure to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

Now we have the three pieces that the A -> Q -> A' pattern needs:

- Q = question, the query generated by the LLM in the ground truth step
- A = answer_orig, the original FAQ answer that served as the source for Q
- A' = answer_llm, the answer produced by our RAG pipeline when it received Q

The evaluation loop compares A' against A. If they carry the same information, the RAG pipeline is doing its job. If A' diverges, the LLM judge in the next section will flag it.


In [53]:
# Before running the full batch, reset the usage we collected while testing:
assistant.reset_usage()

In [54]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, ground_truth, generate_rag_answer)

In [55]:
# answers = []

# for answer_record in results:
#     answers.append(answer_record)
    
# assistant.total_cost()

In [56]:
# df_answers = pd.DataFrame(answers)
# df_answers.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/rag-answers-new.csv", index=False)

# LLM As A Judge

In the previous lesson, we generated RAG answers for our ground truth questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

- the original FAQ answer
- the question generated from that answer
- the answer generated by our RAG pipeline

An LLM judge is another LLM call that compares these three pieces. We ask it whether the RAG answer recovers the same information as the original answer.

It can also explain why an answer is bad.

For example:

- the retrieved document might be wrong
- the answer might miss the key point
- the model might say that it doesn't know

This approach is useful when exact text matching is too strict. The RAG answer doesn't need to copy the FAQ answer word for word. It needs to answer the question with the same key information.

This evaluates the full RAG flow in one pass:

- search: did we retrieve context that contains the answer?
- prompt: did we give the model enough context to answer?
- LLM: did the model produce a useful answer from that context?

If the judge marks an answer as bad, we still need to look at the example. The judge tells us where to investigate. It doesn't replace reading the failing cases.

In [57]:
df_answers = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

Each row has the generated question, the original FAQ answer, and the answer produced by the RAG pipeline.

This is offline evaluation. We can do it because our test dataset came from FAQ records. We know the original answer for every generated question.

In production, we usually don't have that original answer for real user questions. There we can still use an LLM judge. The prompt has to judge only the question and the generated answer. In this lesson, we use the stronger offline setup.

**A->Q->A' evaluation**

We'll compare the RAG answer with the original answer from the FAQ. This checks if the RAG pipeline is producing answers that match the ground truth.

In [58]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [59]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [60]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [61]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [62]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [63]:
# from concurrent.futures import ThreadPoolExecutor

# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, answers, judge_record)

In [64]:
# evaluations = []
# usages = []

# for evaluation, usage in results:
#     evaluations.append(evaluation)
#     usages.append(usage)

# calc_total_price(usages)

# cost: 0.3970770000000002

In [65]:
df_eval = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/rag-evaluations-new.csv")

In [66]:
# df_eval = pd.DataFrame(evaluations)

In [67]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 551/565 = 97.52%


In [68]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
3,What do I need to do to qualify for the certif...,74eb249bbf,bad,The AI answer adds requirements not present in...
97,How can I fix the OpenAI credits issue when ca...,f5df151c59,bad,The ground truth says the error is due to insu...
102,Do I need to add credits to my OpenAI account ...,152af39a53,bad,The AI answer partially captures the key idea ...
103,"After topping up my OpenAI billing, do I need ...",152af39a53,bad,The AI answer captures only one part of the gr...
104,How do I choose a model in responses.create if...,152af39a53,bad,The ground truth says the issue is OpenAI API ...


In [69]:
# df_eval.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/rag-evaluations-new.csv", index=False)

# Agent Evaluation

For RAG, we used the A->Q->A' setup:

- A = original answer in the FAQ
- Q = generated question from this answer
- A' = answer produced by our RAG system

For agents, we use the same setup. A' comes from an agent instead of a fixed RAG pipeline.

We also save the trajectory. Here, the trajectory means only the tool calls the agent made before producing the final answer.

In [70]:
def search(query: str) -> list[dict]:
    """
    Search the FAQ database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question": 1.0, "answer": 2.0, "section": 0.1},
        filter_dict={"course": "llm-zoomcamp"}
    )

In [71]:
agent_tools = Tools()
agent_tools.add_tool(search)

instructions = """
You're a course teaching assistant. Answer student questions based on
the FAQ search results. Use the search tool before answering.
""".strip()

runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    llm_client=OpenAIClient(model="gpt-5.4-mini")
)

The result contains:

- last_message: the final response
- all_messages: the full message history
- cost: the cost of all LLM calls in this run

For this, the trajectory is only the tool calls. We don't need to send the full message history to the judge.

In [72]:
rec = ground_truth[0]

result = runner.loop(prompt=rec["question"])

result.all_messages

[EasyInputMessage(content="You're a course teaching assistant. Answer student questions based on\nthe FAQ search results. Use the search tool before answering.", role='developer', phase=None, type=None),
 EasyInputMessage(content='Can I still join the course if I just found out about it?', role='user', phase=None, type=None),
 ResponseFunctionToolCall(arguments='{"query":"join course late just found out about it enrollment late registration can I still join"}', call_id='call_JL8j3H0FloAfge7964UpVW5P', name='search', type='function_call', id='fc_076a4c9105126da1006a538391c9a48196ab99358b65305c16', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_JL8j3H0FloAfge7964UpVW5P',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit y

In [73]:
def extract_tool_calls(messages):
    tool_calls = []

    for message in messages:
        if isinstance(message, dict):
            continue

        if message.type == "function_call":
            tool_calls.append({
                "name": message.name,
                "arguments": message.arguments,
            })

    return tool_calls


tool_calls = extract_tool_calls(result.all_messages)

tool_calls

[{'name': 'search',
  'arguments': '{"query":"join course late just found out about it enrollment late registration can I still join"}'}]

In [74]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

In [75]:
agent_result = {
    "question": rec["question"],
    "answer_agent": result.last_message,
    "answer_orig": answer_orig,
    "tool_calls": tool_calls,
    "cost": result.cost.total_cost,
    "document": doc_id,
}

agent_result

{'question': 'Can I still join the course if I just found out about it?',
 'answer_agent': 'Yes — you can still join the course even if you just discovered it.\n\nIf you want a certificate, though, you’ll need to submit your project while submissions are still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'tool_calls': [{'name': 'search',
   'arguments': '{"query":"join course late just found out about it enrollment late registration can I still join"}'}],
 'cost': Decimal('0.00094725'),
 'document': '74eb249bbf'}

So far in we built the agent, ran it once, and assembled the first record. The pipeline mirrors the RAG evaluation flow with one extra artifact: the trajectory of tool calls the agent made before producing the final answer.

The record has six fields:

- question: the query from the ground truth (Q)
- answer_orig: the original FAQ answer (A)
- answer_agent: the answer produced by the agent (A')
- tool_calls: the sequence of function calls the agent executed
- cost: the total dollar cost of this run
- document: the doc id for traceability

The sanity check confirms three things at once. The runner loop terminates without error, the extracted tool_calls list is well formed, and the answer_orig lookup returned the expected FAQ text. If any of these had failed on the single query, running the batch would have been a waste.

Next we wrap this logic in a function and run it in parallel on a sample of 50 ground truth records. Agent runs cost more than RAG runs because each query can trigger multiple tool calls, so we limit the batch instead of running the full 395.


In [76]:
def generate_agent_answer(rec):
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    result = runner.loop(prompt=rec["question"])

    tool_calls = extract_tool_calls(result.all_messages)

    answer_record = {
        "question": rec["question"],
        "answer_agent": result.last_message,
        "answer_orig": original_doc["answer"],
        "tool_calls": tool_calls,
        "cost": result.cost.total_cost,
        "document": doc_id,
    }

    return answer_record

In [77]:
# from concurrent.futures import ThreadPoolExecutor


# with ThreadPoolExecutor(max_workers=6) as pool:
#     agent_answers = map_progress(pool, ground_truth[:50], generate_agent_answer)

  0%|          | 0/50 [00:00<?, ?it/s]

In [ ]:
# df_agent = pd.DataFrame(agent_answers)

In [ ]:
df_agent = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/agent-answers.csv")

In [ ]:
# df_agent.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/agent-answers.csv", index=False)

Now we have the same A->Q->A' data as before, plus the tool calls for each agent run.

**Judging answers and trajectories**

A good trajectory is not just "many tool calls". A good trajectory uses the available tools in a way that helps answer the question.

For our search agent, a good trajectory has these properties:

- The search query is relevant to the user question
- The query includes the important keywords from the question
- The agent avoids duplicate searches with the same arguments
- If it searches more than once, the next query is a useful refinement
- It usually uses 1 search call
- 2-3 calls can be okay for harder questions
- More than 3 search calls needs a clear reason
- The tool calls support the final answer
- The agent does not stop too early or keep searching without a reason

Now define a judge output type with two scores:

In [80]:
from pydantic import BaseModel, Field
from typing import Literal

class AgentEvaluation(BaseModel):
    answer_reasoning: str = Field(
        description="Reasoning about whether the final answer is correct."
    )
    answer_score: Literal["good", "bad"] = Field(
        description="'good' if the final answer matches the original answer."
    )
    trajectory_reasoning: str = Field(
        description="Reasoning about whether the tool calls were useful."
    )
    trajectory_score: Literal["good", "bad"] = Field(
        description="'good' if the tool calls were reasonable for the question."
    )

In [81]:
agent_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI agent
4. The tool calls made by the agent

Evaluate two things:

Answer quality:
- Does the agent answer match the original answer?
- It does not need to be word-for-word identical.
- It should contain the same key information.

Trajectory quality:
- Were the search queries relevant to the question?
- Did the queries include important keywords from the question?
- Did the agent avoid duplicate or unnecessary tool calls?
- If it made multiple searches, did the later searches refine the query?
- Was the number of search calls reasonable? Usually 1 is enough, 2-3
  can be okay, and more than 3 needs a clear reason.
- Did the tool calls support the final answer?

Mark answer_score as 'good' if the final answer is correct.
Mark trajectory_score as 'good' if the tool calls were reasonable.
""".strip()

agent_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

Agent Answer:
{answer_agent}

Tool Calls:
{tool_calls}
""".strip()

In [82]:
import json

def evaluate_agent_answer(rec, model="gpt-5.4-mini"):
    tool_calls = rec["tool_calls"]

    if isinstance(tool_calls, str):
        tool_calls = json.loads(tool_calls)

    prompt = agent_judge_prompt.format(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_agent=rec["answer_agent"],
        tool_calls=json.dumps(tool_calls, indent=2),
    )

    result, usage = llm_structured_retry(
        openai_client,
        agent_judge_instructions,
        prompt,
        AgentEvaluation,
        model=model,
    )

    return result, usage

In [83]:
def judge_agent_record(rec):
    agent_eval, usage = evaluate_agent_answer(rec)

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "answer_score": agent_eval.answer_score,
        "answer_reasoning": agent_eval.answer_reasoning,
        "trajectory_score": agent_eval.trajectory_score,
        "trajectory_reasoning": agent_eval.trajectory_reasoning,
    }

    return result, usage

In [84]:
# with ThreadPoolExecutor(max_workers=6) as pool:
#     results = map_progress(pool, agent_answers, judge_agent_record)

  0%|          | 0/50 [00:00<?, ?it/s]

In [85]:
# agent_evaluations = []
# usages = []

# for evaluation, usage in results:
#     agent_evaluations.append(evaluation)
#     usages.append(usage)

# df_agent_eval = pd.DataFrame(agent_evaluations)
# calc_total_price(usages)

#0.05378325

0.05378325

In [ ]:
df_agent_eval = pd.read_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/agent-evaluations.csv")

In [86]:
df_agent_eval["answer_score"].value_counts()

answer_score
good    49
bad      1
Name: count, dtype: int64

In [87]:
df_agent_eval["trajectory_score"].value_counts()

trajectory_score
good    50
Name: count, dtype: int64

In [ ]:
# df_agent_eval.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/data/agent-evaluations.csv", index=False)

# Homework

## Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's
start small and generate questions for just the first 3 pages:

- `01-agentic-rag/lessons/01-intro.md`
- `01-agentic-rag/lessons/02-environment.md`
- `01-agentic-rag/lessons/03-rag.md`

Each call returns the token usage, which most LLM APIs report on the response
object (e.g. `response.usage.input_tokens` / `prompt_tokens`).

What's the average number of input tokens across these 3 calls?

- 140
- **1400**
- 14000
- 140000

In [89]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
print(f"Total de paginas: {len(documents)}")


Total de paginas: 72


In [96]:
# Filter the documents to only include the first 3 lessons
target_files = [
    "01-agentic-rag/lessons/01-intro.md",
    "01-agentic-rag/lessons/02-environment.md",
    "01-agentic-rag/lessons/03-rag.md",
]

first_3_lessons = [d for d in documents if d["filename"] in target_files]
print(f"Encontradas: {len(first_3_lessons)}")
for d in first_3_lessons:
    print(f"  - {d['filename']}")


Encontradas: 3
  - 01-agentic-rag/lessons/01-intro.md
  - 01-agentic-rag/lessons/02-environment.md
  - 01-agentic-rag/lessons/03-rag.md


In [91]:
data_gen_instructions_hw = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

We reuse the structured output helpers from earlier in the notebook.

The `Questions` Pydantic model declares the shape we expect from the LLM: a single field `questions` holding a list of strings. Passing this class to the SDK forces the model to return valid JSON in that exact shape, so the parsing step never fails and we do not have to write our own JSON validation.

In [92]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

The `llm_structured` helper wraps one call. It builds a two-role message list (developer + user), calls `responses.parse` with `text_format=Questions`, and returns the parsed object together with the token usage. A quick test call runs it on a single document to confirm the pipeline works end to end before we scale.


In [94]:
def llm_structured(client, instructions, user_prompt, output_type, model="gpt-5.4-mini"):
    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    response = client.responses.parse(
        model=model,
        input=messages,
        text_format=output_type
    )

    return response.output_parsed, response.usage


The final loop applies the same helper to each of the three target lesson pages. For every page we build the user prompt from `json.dumps(doc)`, run `llm_structured`, and store the generated questions and the token usage in `generated` and `usages_hw`. After the loop finishes, `generated` holds three records, one per lesson page, ready to compute the average input token count that Q1 asks for.


In [97]:
usages_hw = []
generated = []

for doc in first_3_lessons:
    user_prompt = json.dumps(doc)              # redefine a cada iteração
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions_hw,
        user_prompt,
        Questions
    )
    usages_hw.append(usage)
    generated.append({
        "filename": doc["filename"],
        "input_tokens": usage.input_tokens,
        "questions": result.questions,
    })


In [103]:
# Cost breakdown
print("=" * 60)
print("Cost")
print("=" * 60)
for g, u in zip(generated, usages_hw):
    price = calc_price(u)
    print(f"  {g['filename']}: ${price['total_cost']:.6f}")
print(f"\n  Total: ${calc_total_price(usages_hw):.6f}")

# Generated questions per page
for g in generated:
    print("\n" + "=" * 60)
    print(f"{g['filename']}")
    print("=" * 60)
    print(f"Input tokens: {g['input_tokens']}")
    for i, q in enumerate(g['questions'], 1):
        print(f"  {i}. {q}")

# Average input tokens
print("\n" + "=" * 60)
avg_input = sum(g["input_tokens"] for g in generated) / len(generated)
print(f"Average input tokens: {avg_input:.1f}")
print("=" * 60)


Cost
  01-agentic-rag/lessons/01-intro.md: $0.001301
  01-agentic-rag/lessons/02-environment.md: $0.001505
  01-agentic-rag/lessons/03-rag.md: $0.001729

  Total: $0.004534

01-agentic-rag/lessons/01-intro.md
Input tokens: 1020
  1. What is a RAG system supposed to do for an LLM that it can’t do on its own?
  2. Why does the course build the first version of the RAG setup in plain Python instead of using a framework right away?
  3. What are the main limits of LLMs that make RAG useful in the first place?
  4. What kind of example project will this module build to show RAG in action?
  5. What topics are covered in the first part of the module, and how does the second part change the pipeline?

01-agentic-rag/lessons/02-environment.md
Input tokens: 1286
  1. What do I need installed before starting this module, and do I need anything besides Python and Jupyter?
  2. How do I create the project from scratch and which packages should I add first?
  3. Where should I put my LLM API key so

## Q2. First result with text search

Take the first question from the ground truth:

```python
q = ground_truth[0]["question"]
```

After running `text_search` for it, what's the `filename` of the first result?

-  `01-agentic-rag/lessons/01-intro.md`
-  **`01-agentic-rag/lessons/03-rag.md`**
-  `01-agentic-rag/lessons/13-function-calling.md`
-  `01-agentic-rag/lessons/10-rag-next-steps.md`


We generate the ground truth for all 72 lesson pages so we can reuse it across Q2 through Q6. This adapts the same batch pattern from earlier in the notebook but labels each record with `filename` instead of `id`, matching the schema the homework expects.


In [104]:
import json
from concurrent.futures import ThreadPoolExecutor

def map_progress(pool, seq, f):
    results = []

    with tqdm(total=len(seq)) as progress:
        futures = []

        for el in seq:
            future = pool.submit(f, el)
            future.add_done_callback(lambda p: progress.update())
            futures.append(future)

        for future in futures:
            result = future.result()
            results.append(result)

    return results

def generate_ground_truth_hw(doc):
    user_prompt = json.dumps(doc)
    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions_hw,
        user_prompt,
        Questions,
    )
    records = [
        {"question": q, "filename": doc["filename"]}
        for q in out.questions
    ]
    return records, usage


In [105]:

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth_hw)

ground_truth = []
usages_full = []
for records, usage in results:
    ground_truth.extend(records)
    usages_full.append(usage)

print(f"Total questions: {len(ground_truth)}")
print(f"Total cost: ${calc_total_price(usages_full):.4f}")
print(f"First record: {ground_truth[0]}")


  0%|          | 0/72 [00:00<?, ?it/s]

Total questions: 360
Total cost: $0.1139
First record: {'question': 'What is a retrieval-augmented generation system doing for the model, in simple terms?', 'filename': '01-agentic-rag/lessons/01-intro.md'}


In [106]:
df_ground_truth = pd.DataFrame(ground_truth)
df_ground_truth.to_csv("/workspace/llm-zoomcamp-2026/04-evaluation/homework/data/ground_truth_hw.csv", index=False)

In [107]:
df_ground_truth.head()

,question,filename
0,What is a retrieval-augmented generation syste...,01-agentic-rag/lessons/01-intro.md
1,"Why do LLMs sometimes give bad answers, and wh...",01-agentic-rag/lessons/01-intro.md
2,How does the course’s FAQ bot use RAG to answe...,01-agentic-rag/lessons/01-intro.md
3,What gets built in the first part of this modu...,01-agentic-rag/lessons/01-intro.md
4,What changes in the second part when the pipel...,01-agentic-rag/lessons/01-intro.md


We split the 72 lesson pages into overlapping chunks with `chunk_documents(size=2000, step=1000)`. Each chunk keeps the source `filename`, so a retrieved chunk can be traced back to its origin page. This produces 295 chunks total.


In [108]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
print(f"Total chunks: {len(chunks)}")
print(f"Keys in a chunk: {list(chunks[0].keys())}")


Total chunks: 295
Keys in a chunk: ['start', 'content', 'filename']


Then build a text index over the 295 chunks using `minsearch.Index`. Text field is `content`, keyword field is `filename`. Wrapping the search call in a small `text_search` function keeps the interface stable for later reuse in `evaluate`.


In [109]:
from minsearch import Index

text_index = Index(text_fields=['content'], keyword_fields=['filename'])
text_index.fit(chunks)

def text_search(query, num_results=5):
    return text_index.search(query, num_results=num_results)

print("text_index built.")


text_index built.


In [110]:
q = ground_truth[0]["question"]

print(f"Query: {q}")
print(f"Expected filename: {ground_truth[0]['filename']}\n")

results = text_search(q, num_results=5)

print(f"Top-1 filename: {results[0]['filename']}\n")
print("All top 5:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['filename']}")


Query: What is a retrieval-augmented generation system doing for the model, in simple terms?
Expected filename: 01-agentic-rag/lessons/01-intro.md

Top-1 filename: 01-agentic-rag/lessons/03-rag.md

All top 5:
  1. 01-agentic-rag/lessons/03-rag.md
  2. 01-agentic-rag/lessons/03-rag.md
  3. 03-orchestration/lessons/05-rag.md
  4. 04-evaluation/lessons/15-next-steps.md
  5. 01-agentic-rag/lessons/01-intro.md


The question was generated from `01-intro.md`, but `text_search` returned `03-rag.md` at the top. This is not a bug, it is the pedagogical point of the exercise.

The intro page mentions RAG briefly. The dedicated RAG lesson (`03-rag.md`) discusses the same terms with much higher density: retrieval, generation, model, augmented. TF-IDF ranking rewards term density, so the page focused on the topic wins over the page that only touches it in passing.

From a user standpoint, `03-rag.md` is arguably a better answer to the question. It goes deeper on exactly what the question asks. Yet in our ground truth it counts as a miss, because we defined the source page as the only relevant document.

This is the built-in bias of synthetic ground truth with a single relevant document per query. Other pages may be equally relevant or more helpful, and the metric penalizes retrieval for finding them. Hit Rate and MRR measured this way are lower bounds on the real quality of the search. They are still useful for comparing methods on the same dataset, but the absolute numbers understate the actual performance.



## Q3. First result with vector search

After running `vector_search` for the same question, what's the `filename` of
the first result?

- **`01-agentic-rag/lessons/01-intro.md`**
- `01-agentic-rag/lessons/03-rag.md`
- `04-evaluation/lessons/11-evaluation-intro.md`
- `04-evaluation/lessons/12-rag-answers.md`

This question was generated from `01-agentic-rag/lessons/01-intro.md`. Notice
that one method finds the right page at the top and the other doesn't. That's
exactly why we measure across the whole dataset instead of trusting one query.


For vector search we need each chunk represented as a dense vector in the same space as the query. We use `all-MiniLM-L6-v2` from `sentence-transformers`, the same lightweight model we used in module 2. 

In [111]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

chunk_texts = [c["content"] for c in chunks]
X = model.encode(chunk_texts, show_progress_bar=True)
X = np.array(X)

print(f"X shape: {X.shape}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

X shape: (295, 384)


Then index the vectors with `minsearch.VectorSearch`, passing the payload as the same chunks used for text search. The wrapper `vector_search(query, num_results)` encodes the query with the same model, calls the index, and returns the top matches. Keeping the same signature as `text_search` means both are drop-in replacements for each other in `evaluate` later.


In [112]:
from minsearch import VectorSearch

vector_index = VectorSearch()
vector_index.fit(X, chunks)

def vector_search(query, num_results=5):
    v = model.encode(query)
    return vector_index.search(v, num_results=num_results)

print("vector_index built and vector_search function ready.")


vector_index built and vector_search function ready.


reusing the same question from Q2 and running `vector_search`. The `filename` of the top result is the answer to Q3. Because keyword and vector matching optimize different signals, the top-1 here can differ from the top-1 that `text_search` returned.


In [113]:
q = ground_truth[0]["question"]

print(f"Query: {q}")
print(f"Expected filename: {ground_truth[0]['filename']}\n")

results = vector_search(q, num_results=5)

print(f"Top-1 filename: {results[0]['filename']}\n")
print("All top 5:")
for i, r in enumerate(results, 1):
    print(f"  {i}. {r['filename']}")


Query: What is a retrieval-augmented generation system doing for the model, in simple terms?
Expected filename: 01-agentic-rag/lessons/01-intro.md

Top-1 filename: 01-agentic-rag/lessons/01-intro.md

All top 5:
  1. 01-agentic-rag/lessons/01-intro.md
  2. 04-evaluation/lessons/01-intro.md
  3. 04-evaluation/lessons/02-ground-truth.md
  4. 04-evaluation/lessons/11-evaluation-intro.md
  5. 04-evaluation/lessons/02-ground-truth.md


Text search and vector search ranked the same question differently.

Text search picked `03-rag.md` because that page mentions the RAG terms most densely, and TF-IDF rewards term concentration. Vector search compares the query embedding with the chunk embeddings. It rewards semantic similarity, not word overlap. A short overview paragraph in `01-intro.md` may sit closer to the query in embedding space than the technical body of `03-rag.md`, so the intro can come out on top.

Neither result is wrong. They are two views of relevance. The ground truth counts only the source page as a hit, so on this single query text search misses and vector search may or may not hit. That is why we do not judge quality from one query. We run both methods over the full ground truth in the next questions, and let Hit Rate and MRR settle which one works better on this dataset.


Let's evaluate search exactly as in the module, reusing the same functions from the
lecture. We change only the label. Our ground truth uses `filename`, so a result
counts as a hit when a returned chunk's `filename` matches the question's
`filename`, not a document `id`.

As a reminder, these functions do the following:

- `compute_relevance` runs search for a question and returns a list of 0s and 1s
- `hit_rate` is the fraction of questions where the correct page appears in the results
- `mrr` (Mean Reciprocal Rank) also rewards finding the page near the top
- `evaluate` runs a search function over the whole ground truth and returns both metrics


## Q4. Evaluating text search

Evaluate `text_search` on the ground truth data.

What's the Hit Rate?

- 0.55
- 0.66
- **0.76**
- 0.88


The `evaluate` function from the module compares each retrieved chunk's `id` against the ground truth's `document` field. Our ground truth uses `filename`, so we make a small variant that reads the `filename` field on both sides. Everything else stays the same.


In [114]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [115]:
from tqdm.auto import tqdm

def compute_relevance_filename(q, search_function):
    correct = q["filename"]
    results = search_function(query=q["question"])
    return [int(r["filename"] == correct) for r in results]

def compute_relevance_total_filename(ground_truth, search_function):
    relevance_total = []
    for q in tqdm(ground_truth):
        relevance = compute_relevance_filename(q, search_function)
        relevance_total.append(relevance)
    return relevance_total

def evaluate_filename(ground_truth, search_function):
    relevance_total = compute_relevance_total_filename(ground_truth, search_function)
    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

print("evaluate_filename ready.")


evaluate_filename ready.


We run `evaluate_filename` with `text_search` over the full 360-question ground truth. This calls the search function once per question, builds a 360x5 relevance matrix, and reduces it to two numbers.


In [116]:
metrics_text = evaluate_filename(ground_truth, text_search)
print(f"text_search: hit_rate={metrics_text['hit_rate']:.4f}, mrr={metrics_text['mrr']:.4f}")


  0%|          | 0/360 [00:00<?, ?it/s]

text_search: hit_rate=0.8028, mrr=0.6262



## Q5. Evaluating vector search

Now evaluate `vector_search` - the part we left for the homework, since the
module only evaluated keyword search.

What's the MRR?

- 0.35
- 0.45
- 0.55
- **0.65**


In [118]:
metrics_vector = evaluate_filename(ground_truth, vector_search)
print(f"vector_search: hit_rate={metrics_vector['hit_rate']:.4f}, mrr={metrics_vector['mrr']:.4f}")


  0%|          | 0/360 [00:00<?, ?it/s]

vector_search: hit_rate=0.8083, mrr=0.6166


- Text search: hit_rate=0.8028, mrr=0.6262
- Vector search: hit_rate=0.8083, mrr=0.6166

Basically tied. Vector finds the page a bit more often, text puts it higher when it finds. Neither wins.

Q6 tests if fusing both with RRF beats each one alone. If hybrid MRR jumps above 0.62, they were catching different queries. If it stays around 0.62, they were catching the same ones.



## Q6. Tuning hybrid search

The `k` constant in RRF controls how much the top ranks matter. A smaller `k`
sharpens the gap between positions, so being at the top of a list counts for
more. The RRF paper uses 60 as a default, but the best value depends on the data
- so let's measure it.

Evaluate `hybrid_search` over the full ground truth dataset for `k` values 1,
50, 100, and 200. Compare the MRR values for these runs.

Which `k` gives the best MRR?

- 1
- **50**
- 100
- 200

> Several values of `k` may give the same MRR. If there's a tie, pick the
> smallest `k`.


Reciprocal Rank Fusion merges two ranked lists using only positions. Each doc gets a score of 1/(k + rank) from each list, and we sort by the sum. Small k sharpens the top-rank bonus, big k flattens it.


In [119]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}
    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc
    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

def hybrid_search(query, k=60, num_results=5):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k, num_results=num_results)

print("hybrid_search ready.")


hybrid_search ready.


We evaluate hybrid_search on the full ground truth for k = 1, 50, 100, 200. Since evaluate_filename expects a search function with the same signature as text_search, we wrap hybrid_search in a lambda that fixes k for each run.


In [121]:
results_hybrid = {}

for k in [1, 50, 100, 200]:
    print(f"\nEvaluating k={k}...")
    hybrid_fn = lambda query, k=k: hybrid_search(query, k=k, num_results=5)
    metrics = evaluate_filename(ground_truth, hybrid_fn)
    results_hybrid[k] = metrics
    print(f"  k={k}: hit_rate={metrics['hit_rate']:.4f}, mrr={metrics['mrr']:.4f}")



Evaluating k=1...


  0%|          | 0/360 [00:00<?, ?it/s]

  k=1: hit_rate=0.8639, mrr=0.6727

Evaluating k=50...


  0%|          | 0/360 [00:00<?, ?it/s]

  k=50: hit_rate=0.8722, mrr=0.6950

Evaluating k=100...


  0%|          | 0/360 [00:00<?, ?it/s]

  k=100: hit_rate=0.8722, mrr=0.6950

Evaluating k=200...


  0%|          | 0/360 [00:00<?, ?it/s]

  k=200: hit_rate=0.8722, mrr=0.6950


In [122]:
best_k = None
best_mrr = -1

for k, m in results_hybrid.items():
    if m["mrr"] > best_mrr:
        best_mrr = m["mrr"]
        best_k = k

print(f"\nSummary:")
for k, m in results_hybrid.items():
    print(f"  k={k}: mrr={m['mrr']:.4f}, hit_rate={m['hit_rate']:.4f}")

print(f"\nBest k: {best_k} (MRR={best_mrr:.4f})")



Summary:
  k=1: mrr=0.6727, hit_rate=0.8639
  k=50: mrr=0.6950, hit_rate=0.8722
  k=100: mrr=0.6950, hit_rate=0.8722
  k=200: mrr=0.6950, hit_rate=0.8722

Best k: 50 (MRR=0.6950)


- Text search: hit_rate=0.8028, mrr=0.6262
- Vector search: hit_rate=0.8083, mrr=0.6166
- Hybrid (k=50): hit_rate=0.8722, mrr=0.6950

Text and vector are tied on their own. Hybrid beats both by around 5 points on Hit Rate and 7 points on MRR. That gain proves the two methods were catching different queries. If they were catching the same ones, RRF would have nothing to combine and the numbers would stay flat.

On the tuning of k: k=1 was worse because it makes top-1 dominate too much and gives no weight to positions 2 through 5. From k=50 upwards the score is identical (0.6950 MRR, 0.8722 Hit Rate), so we pick the smallest per the homework rule. Anywhere in the 50 to 200 range, RRF is stable on this dataset.

neither text nor vector is enough alone. Hybrid with a moderate k gives the best of both without any additional tuning.
